# 04 — Evaluation

Loads a trained model and computes the full evaluation metric suite on the test split. For CLI use, prefer `python evaluate.py`.

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import tensorflow as tf
from config import CONFIG
from src.dataset import PneumoniaDataset
from src.evaluator import Evaluator

In [ ]:
model_path = CONFIG.paths.best_model_path
if model_path.exists():
    model = tf.keras.models.load_model(model_path)
else:
    model = None
    print(f'No trained model found at {model_path}. Run train.py first.')

## Run predictions on the test set

In [ ]:
if model is not None:
    test_data = PneumoniaDataset(CONFIG.paths.test_dir, image_size=CONFIG.data.image_size, batch_size=CONFIG.data.batch_size, cache=False)
    test_ds = test_data.build(training=False)
    y_prob = model.predict(test_ds, verbose=1).ravel()
    y_true = np.array(test_data.labels)[: len(y_prob)]

## Metrics, confusion matrix, ROC, PR curve

In [ ]:
if model is not None:
    evaluator = Evaluator(class_names=CONFIG.data.class_names, output_dir=CONFIG.paths.outputs_dir)
    report = evaluator.full_report(y_true, y_prob)
    for k, v in report['metrics'].items():
        print(f'{k}: {v}')